In [2]:
# =========================================================
# BERT4Rec with RecBole on MovieLens-1M (KAGGLE NOTEBOOK)
# SINGLE CELL VERSION
# Auto-download + preprocess + RecBole format + train
# =========================================================

# ================= INSTALL =================
!pip install -q recbole ray pandas

# ================= IMPORTS =================
import os
import zipfile
import urllib.request
import pandas as pd
from recbole.quick_start import run_recbole

# ================= CONFIG =================
DATA_DIR = '/kaggle/working/ml-1m'
ZIP_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
ZIP_PATH = '/kaggle/working/ml-1m.zip'
RATINGS_PATH = os.path.join(DATA_DIR, 'ml-1m', 'ratings.dat')
DATASET_NAME = 'movielens1m'

# ================= DOWNLOAD DATA =================
if not os.path.exists(RATINGS_PATH):
    print("Downloading MovieLens 1M ...")
    urllib.request.urlretrieve(ZIP_URL, ZIP_PATH)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(DATA_DIR)
    print("Download & extraction done")
else:
    print("MovieLens already exists")

# ================= PREPROCESS =================
print("Preparing RecBole format ...")
df = pd.read_csv(
    RATINGS_PATH,
    sep='::', engine='python',
    names=['user_id', 'item_id', 'rating', 'timestamp']
)

# RecBole only needs interactions
ratings_csv = os.path.join(DATA_DIR, 'ratings.csv')
df[['user_id', 'item_id', 'timestamp']].to_csv(ratings_csv, index=False)
print("ratings.csv saved for RecBole")

# ================= TRAIN CONFIG =================
config_dict = {
    'data_path': [DATA_DIR],
    'dataset': DATASET_NAME,
    'USER_ID_FIELD': 'user_id',
    'ITEM_ID_FIELD': 'item_id',
    'TIME_FIELD': 'timestamp',
    'load_col': {
        'inter': ['user_id', 'item_id', 'timestamp']
    },
    'max_seq_length': 50,
    'TRAIN_BATCH_SIZE': 256,
    'eval_batch_size': 256,
    'epochs': 10,
    'embedding_size': 64,
    'hidden_size': 64,
    'inner_size': 256,
    'n_layers': 2,
    'n_heads': 2,
    'mask_ratio': 0.2,
    'train_neg_sample_args': None,
    'metrics': ['Recall', 'MRR', 'NDCG', 'Hit', 'Precision'],
    'topk': 10,
    'device': 'cuda' if os.system('nvidia-smi > /dev/null 2>&1') == 0 else 'cpu'
}

# ================= TRAIN =================
print("\n===== START TRAINING BERT4REC =====")
run_recbole(
    model='BERT4Rec',
    dataset=DATASET_NAME,
    config_dict=config_dict
)
print("===== TRAINING FINISHED =====")

/home/nbx/Desktop/LengthAdaptiveFusion-GNN-Transformer/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-07 17:11:45,535	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


ImportError: Can't import ray.tune as some dependencies are missing. Run `pip install "ray[tune]"` to fix.